# 02 — Contrats de données — les messages, les annotations, et le lien entre eux

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Lire les schémas comme des **contrats exécutables** : colonnes, types, domaines de valeurs.
1. Casser chaque contrat volontairement : un schéma qu'on n'a jamais vu échouer ne protège rien.
1. Vérifier le **lien** entre les deux tables : `surface == text[start:end]`, pas de chevauchement, splits alignés.
1. Valider la table de prédictions, entrée de la chaîne d'inférence.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 2. Les schémas, colonne par colonne

Un schéma Pandera est un objet que l'on peut **afficher** : c'est ce qui le rend auditable. Les descriptions viennent du code du projet, pas d'un document à part.

In [ ]:
from pandera.errors import SchemaError, SchemaErrors

from src.data.schemas import MessagesSchema, SpansSchema, validate_messages, validate_spans


def contract_table(schema: type) -> pd.DataFrame:
    """Render a Pandera schema as the table a reader can audit."""
    columns = schema.to_schema().columns
    return pd.DataFrame(
        {
            "colonne": list(columns),
            "type": [str(column.dtype) for column in columns.values()],
            "contraintes": [
                ", ".join(check.name for check in column.checks) or "—"
                for column in columns.values()
            ],
            "rôle": [str(column.description or "—") for column in columns.values()],
        }
    )


display(contract_table(MessagesSchema))
display(contract_table(SpansSchema))

**Ce qu'il faut retenir**

- Le lien entre les tables n'est pas une convention : `msg_id` est la clé de jointure, et son format (`MSG-0001`) est contractuel.
- `split` est une **colonne**, pas un paramètre d'entraînement : c'est ce qui rend deux exécutions comparables.
- Le drapeau `holdout` est publié avec les données : la dégradation sur les surfaces réservées se mesure donc au lieu de se commenter.

## 3. Le contrat des messages

Six colonnes suffisent à décrire une ligne : l'identifiant, le texte, le canal, le style, deux compteurs et la date. `strict = True` signifie qu'une colonne non déclarée est une **erreur**, pas un détail.

In [ ]:
VALIDATED_MESSAGES = validate_messages(DOCUMENTS)
print(
    f"messages valides : {len(VALIDATED_MESSAGES)} lignes, "
    f"{len(VALIDATED_MESSAGES.columns)} colonnes"
)

casse = DOCUMENTS.copy()
casse.loc[casse.index[0], "msg_id"] = "1234"  # format d'identifiant invalide
casse.loc[casse.index[1], "text"] = "court"  # texte sous la longueur minimale
casse.loc[casse.index[2], "canal"] = "pigeon"  # canal non déclaré
try:
    validate_messages(casse)
except SchemaError as error:
    print("SchemaError attendue :", str(error).splitlines()[0])

In [ ]:
casse_lazy = DOCUMENTS.copy()
casse_lazy.loc[casse_lazy.index[0], "msg_id"] = "1234"
casse_lazy.loc[casse_lazy.index[1], "text"] = "court"
casse_lazy.loc[casse_lazy.index[2], "canal"] = "pigeon"
try:
    validate_messages(casse_lazy, lazy=True)
except SchemaErrors as errors:
    print(
        f"lazy=True : {errors.failure_cases['column'].nunique()} colonnes fautives signalées "
        f"en une seule passe"
    )
    display(errors.failure_cases[["column", "check", "failure_case"]].head(6))

**Ce qu'il faut retenir**

- `lazy=True` collecte **toutes** les erreurs d'un coup : sur un fichier de 1 200 lignes, corriger une colonne par exécution coûte plus cher que la validation.
- Le message d'erreur nomme la ligne fautive : c'est ce qui rend un contrat utilisable par quelqu'un qui n'a pas écrit le schéma.
- Le contrat est appliqué **à l'écriture** par le chargeur : un corpus non conforme ne touche jamais le disque.

## 4. Le contrat des annotations, et le lien entre les tables

Une annotation est une paire d'offsets **dans un message**. Le contrat ne peut donc pas la valider seule : il faut le texte pour vérifier que `surface == text[start:end]`, et l'appartenance au même split.

In [ ]:
from src.data.schemas import corpus_violations

VALIDATED_SPANS = validate_spans(SPANS)
PROBLEMES = corpus_violations(DOCUMENTS, SPANS)
print(f"annotations valides : {len(VALIDATED_SPANS)}")
print("violations du corpus :", PROBLEMES or "aucune")

decalage = SPANS.copy()
decalage.loc[decalage.index[0], "start"] = decalage.loc[decalage.index[0], "start"] + 1
print("offsets décalés d'un caractère :", corpus_violations(DOCUMENTS, decalage))

inconnu = SPANS.copy()
inconnu.loc[inconnu.index[0], "label"] = "numero_de_dossier"
print("type inventé :", corpus_violations(DOCUMENTS, inconnu))

In [ ]:
chevauchements = []
for message_id, groupe in SPANS.groupby("msg_id"):
    bornes = [tuple(borne) for borne in groupe[["start", "end"]].to_numpy()]
    for index, (start, end) in enumerate(bornes):
        for autre_start, autre_end in bornes[index + 1 :]:
            if min(end, autre_end) - max(start, autre_start) > 0:
                chevauchements.append((message_id, (start, end), (autre_start, autre_end)))
print(f"messages du corpus : {DOCUMENTS['msg_id'].nunique()}")
print(f"chevauchements de mentions : {len(chevauchements)} (aucun n'est une information métier)")

**Ce qu'il faut retenir**

- `surface == text[start:end]` est vérifié sur le corpus entier : sans ce contrôle, une annotation décalée d'un caractère passerait inaperçue jusqu'à ce qu'un rapport explique une F1 inexplicable.
- Un type inventé est refusé : il deviendrait sinon une classe publiée du rapport, que personne ne saurait interpréter.
- Deux mentions qui se chevauchent dans un même message ne sont pas une ambiguïté riche : c'est une erreur d'annotation.

## 5. Le contrat des prédictions

La table de prédictions est l'entrée de la chaîne d'inférence : mêmes offsets, un type, une **provenance** (`regle` ou `modele`) et une confiance bornée. On la produit ici avec le modèle à règles seules, qui n'a rien à apprendre.

In [ ]:
from src.data.schemas import validate_predictions
from src.inference.predictor import EntityPredictor
from src.models import build_model

rules_model = build_model(CONFIG, algorithm="gazetteer", params={})
rules_model.fit(TRAIN_DOCUMENTS, TRAIN_SPANS)
predictor = EntityPredictor(rules_model, config=CONFIG.model_dump(), id_column=ID_COLUMN)
PREDICTIONS = predictor.predict(VAL_DOCUMENTS, spans=VAL_SPANS)
print(f"prédictions : {len(PREDICTIONS)} mentions sur {len(VAL_DOCUMENTS)} messages")
display(PREDICTIONS.head(5))
display(validate_predictions(PREDICTIONS).head(3))

In [ ]:
casse_pred = PREDICTIONS.copy()
casse_pred.loc[casse_pred.index[0], "confidence"] = 1.4  # hors [0, 1]
casse_pred.loc[casse_pred.index[1], "source"] = "intuition"  # provenance non déclarée
casse_pred.loc[casse_pred.index[2], "label"] = "facture"  # type inconnu du projet
try:
    validate_predictions(casse_pred)
except SchemaError as error:
    print("SchemaError attendue :", str(error).splitlines()[0])

**Ce qu'il faut retenir**

- La provenance est contractuelle : une mention corroborée par une règle et une mention proposée par le modèle ne se relisent pas de la même façon.
- La confiance est bornée dans [0, 1] par le schéma — c'est un **niveau**, et le notebook 04 montre comment le lire avant de l'utiliser comme seuil.
- La table de prédictions est validée à l'écriture par le chargeur : la chaîne d'inférence ne peut pas publier une table non conforme.

## Synthèse

| Contrat | Objet | Ce qu'il empêche |
| --- | --- | --- |
| Messages | `MessagesSchema` | identifiants dupliqués, texte tronqué, canal inventé |
| Annotations | `SpansSchema` | offsets inversés, type inventé, surface vide |
| Corpus | `corpus_violations` | `surface != text[start:end]`, mention hors de son split, chevauchement |
| Prédictions | `PredictedMentionsSchema` | provenance inconnue, confiance hors bornes |

**Suite** : `03_preprocessing.ipynb` transforme ces tables en traits mesurables.